# Session 9 — State of the art: a niche for a product that does not exist yet

> **Work in your own copy** (from day 1): *File → Save a copy in Drive*, then *File → Move* it into **DS403 students → your folder** (your first and last name). Work only there: your teachers follow your progress from that folder. In the exercise cells, **stop a little and think** before you type, and solve them **without consulting generative systems**.


**DS403 · Thu 29 Oct 2026 · 09:00–12:00** · the deck: *Session 9 · Finding a niche with data and agents* (PDF on Drive,
`slides/session-09.pdf`). Every block below names its slides; every Part of the deck names its block here.

**The product (it does not exist):** *Ritmo Coach*, an AI habit coach sold as a paid add-on inside Ritmo, the habit app
whose data you have used since session 1.
**The question:** which niche do we launch first, how big is it, **and how sure are we?**

| Block | What you do | Data kind | Deck |
|---|---|---|---|
| 0 | the brief and the evidence ledger | — | Part 1 |
| 1 | open data on spending and population (Eurostat, live) | **open** | Part 2 |
| 2 | Ritmo's funnel and its price test | **partial** | Part 2 |
| 2b | conditional probability and Bayes on the funnel | partial | Part 1 |
| 3 | segment the audience | partial | Part 2 |
| 3b | reweighting: your sample is not your market (our group, Spain, Ritmo) | open + partial | Part 3 |
| 4 | simulated users: shape, never level | **simulated** | Part 4 |
| 5 | competitors, segmented | research | Part 2 |
| 5b | features from existing apps (App Store descriptions and reviews) | research | Part 2 |
| 6 | what people say (Hacker News, live) | **conversations** | Part 2 |
| 6b | which features matter to which user group | research | Part 2 |
| 7 | automated analysis: an agent writes, the notebook checks | — | Part 5 |
| 8 | fuse it all: a provable niche estimate | all | Part 6 |
| 8b | fusion by statistical matching: overlap, bounds, a bridge | all | Part 6 |
| 8c | the key user journey, simulated per group | partial + simulated | Part 6 |
| 8d | evidence synthesis: weak supervision, Bayesian synthesis, triangulation, Dempster–Shafer | all | Part 7 |
| 9 | niche variants | all | Part 8 |
| 10 | steer: what to research next, and update | research | Part 8 |
| 11 | your own product | — | Lab |

**Start with the cheat sheet**, `examples/session-09/CHEATSHEET.pdf` (or `.md`): your project's niche in thirteen steps, the
formulas and the thresholds. The instruction files for your own Claude or ChatGPT are beside it (README there): drop
them into a chat **outside** this pipeline, or call a model with them **inside** it (block 7).

In [ ]:
#@title Course setup — run this first (on Colab it prepares Drive, pandas and the data; on a laptop it does nothing)
import os, shutil, subprocess, sys
from pathlib import Path

COURSE_FOLDER = "261019 HSIT Generative Python product development"   # the shared course folder on Google Drive
COURSE_FOLDER_ID = "1z3bZ5zoE6mE7-oyLk1gmBDXGrEVFTlXd"

if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")
    from importlib.metadata import version
    if int(version("pandas").split(".")[0]) < 3:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pandas>=3.0,<3.1", "pyarrow"], check=True)
        if "pandas" in sys.modules:   # Colab loaded the old pandas already: only a restart replaces it
            raise SystemExit("pandas 3 is installed. Now: Runtime → Restart session, then run this cell again.")
    import pandas as pd
    home = Path("/content/drive/MyDrive/ds403")
    data = home / "data"
    if not (data / "events.csv").exists():
        drive_root = Path("/content/drive")
        # By the folder's id, never by searching a whole Drive (slow, and it depends on where you keep things):
        # the shortcut in My Drive (wherever you put it, Drive lists its target by id), or the folder itself in My Drive.
        src = next((c for c in (drive_root / ".shortcut-targets-by-id" / COURSE_FOLDER_ID / COURSE_FOLDER / "data",
                                drive_root / "MyDrive" / COURSE_FOLDER / "data") if (c / "events.csv").exists()), None)
        data.mkdir(parents=True, exist_ok=True)
        if src:
            for f in ("events.csv", "audience.csv"):
                shutil.copy(src / f, data / f)
        else:
            # No shortcut: read the two files straight from the course folder (Google asks once for permission).
            from google.colab import auth
            from googleapiclient.discovery import build
            from googleapiclient.http import MediaIoBaseDownload
            auth.authenticate_user()
            api = build("drive", "v3")
            def children(parent, extra=""):
                return api.files().list(q=f"'{parent}' in parents and trashed = false{extra}", fields="files(id,name)",
                                        supportsAllDrives=True, includeItemsFromAllDrives=True).execute()["files"]
            files = []
            try:   # the folder named 'data' that holds the files (Drive allows two folders with one name)
                for d in children(COURSE_FOLDER_ID, " and name = 'data'"):
                    files = children(d["id"])
                    if any(f["name"] == "events.csv" for f in files):
                        break
                    files = []
            except Exception:
                files = []
            if not files:
                raise FileNotFoundError(f"You cannot open the course folder '{COURSE_FOLDER}' yet: ask your teacher to "
                                        "share it with you, then run this cell again.")
            for f in files:
                if f["name"] in ("events.csv", "audience.csv"):
                    with open(data / f["name"], "wb") as out:
                        down = MediaIoBaseDownload(out, api.files().get_media(fileId=f["id"]))
                        done = False
                        while not done:
                            _, done = down.next_chunk()
        print("The course data is copied to", data)
    # Work from a local folder beside a link to your data on Drive: '../data' then means your ds403/data, exactly as
    # on a laptop (inside the Drive mount itself, '..' does not resolve), and what a notebook saves still lands on Drive.
    local = Path("/content/ds403")
    (local / "work").mkdir(parents=True, exist_ok=True)
    if not (local / "data").exists():
        (local / "data").symlink_to(data, target_is_directory=True)
    os.chdir(local / "work")
    print("Ready on Colab · pandas", pd.__version__, "· data in", data)


In [ ]:
from __future__ import annotations
import json, re, urllib.request, urllib.parse
from datetime import date
from pathlib import Path
import numpy as np
import pandas as pd

pd.set_option("display.width", 140)
rng = np.random.default_rng(9)
TODAY = date.today().isoformat()
DATA = Path("../data")                          # events.csv and audience.csv (the course setup cell puts them there)
_drive = Path("/content/drive")                   # on Colab: the course folder, by its id (shortcut) or in My Drive
_cf, _cid = globals().get("COURSE_FOLDER", ""), globals().get("COURSE_FOLDER_ID", "")
EX = next((p for p in (Path("../examples/session-09"), Path("examples/session-09"),
                       _drive / ".shortcut-targets-by-id" / _cid / _cf / "examples" / "session-09",
                       _drive / "MyDrive" / _cf / "examples" / "session-09") if _cf is not None and p.exists()), Path("."))
print("data:", DATA.resolve(), "| examples:", EX.resolve())

## Block 0 · The brief and the evidence ledger (deck: Part 1)

A **provable** estimate is one where anyone can walk from the answer back to every number it used: where it came
from, when, what kind of evidence it is, and how wide it is. So before any analysis we open a **ledger**. Every number
that enters the estimate goes through `fact()`; the fusion in block 8 refuses numbers that are not in it.

Five kinds of evidence, in falling order of trust for a **level** (how many, how much):
`open` (official statistics) · `partial` (our own users' behaviour) · `research` (a published source, a provider's
report, a deep-research answer you checked) · `assumption` (a guess, said out loud) · `simulated` (LLM or agent-based
users: allowed to set a **shape**, such as which niche ranks higher, never a level).

In [ ]:
LEDGER: list[dict] = []
KINDS = ("open", "partial", "research", "assumption", "simulated")

def fact(name: str, low: float, high: float, unit: str, kind: str, source: str, note: str = "") -> tuple[float, float]:
    """Put one number (as a range) into the ledger and return it. A point estimate is a range with low == high."""
    assert kind in KINDS, kind
    assert low <= high, (name, low, high)
    LEDGER.append(dict(name=name, low=low, high=high, unit=unit, kind=kind, source=source, note=note, at=TODAY))
    return low, high

def ledger() -> pd.DataFrame:
    return pd.DataFrame(LEDGER).drop_duplicates("name", keep="last").set_index("name")

NICHES = pd.DataFrame([
    # niche,          age bands in Ritmo, the job to be done
    ("students",      ["18-24"],          "keep a study routine through exams"),
    ("remote_workers",["25-34", "35-44"], "protect focus and stop work spilling into the evening"),
    ("parents",       ["35-44", "45-54"], "run the family's routines without nagging"),
    ("health_55",     ["55+"],            "keep health habits a doctor asked for"),
], columns=["niche", "age_bands", "job"]).set_index("niche")
NICHES

**Your notes · Block 0** — what you think about this task: what was clear, what was hard or confusing, what you would do differently, and your feedback for the teachers. Your own words, a few lines are enough; your teachers read them in your folder.

*Write below this line (double-click this cell, then Shift+Enter):*

…


## Block 1 · Open data: how many people, and what they spend (deck: Part 2, *Open data*)

Eurostat's API needs no key. We read (a) Spain's population by age group, 2024, and (b) what a Spanish household spends
on recreation and culture (Household Budget Survey, 2020, in purchasing-power standards). If the API is down, the
notebook falls back to the snapshot taken on 29 Oct 2026 (`eurostat_snapshot_2026-10-29.json`) and says so.

In [ ]:
EUROSTAT = "https://ec.europa.eu/eurostat/api/dissemination/statistics/1.0/data/"
QUERIES = {
    "population_es_2024": "demo_pjangroup?format=JSON&geo=ES&sex=T&time=2024&lang=EN",
    "household_spend_es": "hbs_exp_t121?format=JSON&geo=ES&lang=EN&time=2020&coicop=CP09&coicop=CP093&coicop=CP094&coicop=CP0941&coicop=CP10",
}

def jsonstat(d: dict, dim: str) -> pd.Series:
    """One dimension of a Eurostat JSON-stat answer as a Series (label → value); the other dimensions have size 1."""
    cat = d["dimension"][dim]["category"]
    pos = {str(i): k for k, i in cat["index"].items()}
    return pd.Series({cat["label"][pos[i]]: v for i, v in d["value"].items() if i in pos}, name=d["label"])

try:
    raw = {k: json.load(urllib.request.urlopen(EUROSTAT + q, timeout=20)) for k, q in QUERIES.items()}
    raw = {k: v | {"dimension": {kk: v["dimension"][kk] for kk in v["id"]}} for k, v in raw.items()}
    for v in raw.values():
        for kk in v["id"]:
            v["dimension"][kk] = {"category": v["dimension"][kk]["category"]} if "category" in v["dimension"][kk] else {"category": v["dimension"][kk]}
    src = "Eurostat API, live " + TODAY
except Exception as e:
    snap = json.loads((EX / "eurostat_snapshot_2026-10-29.json").read_text())
    raw = {k: snap[k] | {"dimension": {kk: {"category": c} for kk, c in snap[k]["dimension"].items()}} for k in QUERIES}
    src = f"Eurostat snapshot of 2026-10-29 (live call failed: {type(e).__name__})"
pop = jsonstat(raw["population_es_2024"], "age")
spend = jsonstat(raw["household_spend_es"], "coicop")
print(src); display(pop.head(8)); display(spend)

In [ ]:
# Ritmo's age bands from Eurostat's 5-year groups (adults only; 55+ stops at 74 here: say so in the ledger).
groups = {"18-24": [("From 15 to 19 years", 2/5), ("From 20 to 24 years", 1)],
          "25-34": [("From 25 to 29 years", 1), ("From 30 to 34 years", 1)],
          "35-44": [("From 35 to 39 years", 1), ("From 40 to 44 years", 1)],
          "45-54": [("From 45 to 49 years", 1), ("From 50 to 54 years", 1)],
          "55+":   [("From 55 to 59 years", 1), ("From 60 to 64 years", 1), ("From 65 to 69 years", 1), ("From 70 to 74 years", 1)]}
band_pop = pd.Series({b: sum(pop[g] * w for g, w in parts) for b, parts in groups.items()})
for b, n in band_pop.items():
    fact(f"population_{b}", n, n, "people", "open", src + " demo_pjangroup", "55+ is 55–74" if b == "55+" else "")
rec = spend["Recreation and culture"]
fact("household_recreation_pps", rec * 0.9, rec * 1.1, "PPS/household/year", "open", src + " hbs_exp_t121 CP09 (2020)", "±10%: 2020 was a pandemic year")
band_pop.map("{:,.0f}".format)

**Your notes · Block 1** — what you think about this task: what was clear, what was hard or confusing, what you would do differently, and your feedback for the teachers. Your own words, a few lines are enough; your teachers read them in your folder.

*Write below this line (double-click this cell, then Shift+Enter):*

…


## Block 2 · Partial data: Ritmo's funnel and its price test (deck: Part 2, *Partial data*)

We never observed anyone buying *Ritmo Coach*. We did observe Ritmo's users meeting a **price**: `view_pricing` (screen
`pricing_a` at 3.99 or `pricing_b` at 4.99) → `start_trial` → `subscribe`. That is partial evidence of willingness to
pay, by age band. Partial, because these people already use a habit app (selection), and because the price was for
Ritmo premium, not for a coach.

In [ ]:
aud = pd.read_csv(DATA / "audience.csv")
ev = pd.read_csv(DATA / "events.csv")
first_price = (ev[ev.event == "view_pricing"].sort_values("ts").groupby("user_id").screen.first().rename("arm"))
did = ev.pivot_table(index="user_id", columns="event", values="ts", aggfunc="count", fill_value=0)
u = aud.set_index("user_id").join(did).join(first_price)
saw = u[u.view_pricing > 0].copy()
saw["trial"] = saw.start_trial > 0
saw["paid"] = saw.subscribe > 0
funnel = saw.groupby(["age_band", "arm"]).agg(viewers=("paid", "size"), trials=("trial", "sum"), paid=("paid", "sum"))
funnel["paid_rate"] = funnel.paid / funnel.viewers
funnel.round(3)

In [ ]:
# A Beta posterior per age band and price: paid ~ Binomial(viewers, p), p ~ Beta(1, 1).
from scipy import stats
post = {}
for (band, arm), r in funnel.iterrows():
    d = stats.beta(1 + r.paid, 1 + r.viewers - r.paid)
    post[(band, arm)] = d
    lo, hi = d.ppf([0.05, 0.95])
    fact(f"paid_rate_{band}_{arm}", lo, hi, "share of pricing viewers", "partial", "Ritmo events.csv, view_pricing → subscribe")
pd.DataFrame({k: v.ppf([0.05, 0.5, 0.95]) for k, v in post.items()}, index=["p5", "p50", "p95"]).T.round(3)

**Read it:** where the 3.99 and 4.99 intervals overlap, the price test did not separate the prices for that band. That
is a finding, not a failure: it tells block 10 where more data would pay off.

**Your notes · Block 2** — what you think about this task: what was clear, what was hard or confusing, what you would do differently, and your feedback for the teachers. Your own words, a few lines are enough; your teachers read them in your folder.

*Write below this line (double-click this cell, then Shift+Enter):*

…


## Block 2b · Conditional probability and Bayes on the funnel (deck: Part 1, *Every rate is conditional*, *Bayes*)

Every rate in this notebook has a "given". Four formulas carry the whole estimate:

**Conditional probability** $\;P(A \mid B) = \dfrac{P(A \cap B)}{P(B)}$

**The chain rule (a funnel)** $\;P(\text{pay}) = P(\text{price})\,P(\text{trial} \mid \text{price})\,P(\text{pay} \mid \text{trial}, \text{price})$

**Total probability (segments)** $\;P(\text{pay}) = \sum_s P(\text{pay} \mid s)\,P(s)$

**Bayes** $\;P(H \mid D) = \dfrac{P(D \mid H)\,P(H)}{\sum_h P(D \mid h)\,P(h)}$, and for a rate with a Beta prior:
$p \sim \text{Beta}(\alpha, \beta)$, $k$ of $n$ paid $\Rightarrow$ $p \mid k, n \sim \text{Beta}(\alpha + k,\ \beta + n - k)$.

Route A in block 8 is the chain rule: $\text{SAM} = N \cdot P(\text{app}) \cdot P(\text{price} \mid \text{app}) \cdot
P(\text{pay} \mid \text{price}, \text{app}, \text{niche}) \cdot € \cdot 12$. Multiplying shares is only right when each
is conditional on the steps before it.

In [ ]:
# 1. The chain rule on Ritmo: the product of conditionals equals the direct count.
n_all = len(u)
p_price = (u.view_pricing > 0).mean()
p_trial_given_price = saw.trial.mean()
p_pay_given_trial_price = saw[saw.trial].paid.mean()
chain = p_price * p_trial_given_price * p_pay_given_trial_price
direct = ((u.view_pricing > 0) & (u.start_trial > 0) & (u.subscribe > 0)).mean()
print(f"P(price)={p_price:.3f} · P(trial|price)={p_trial_given_price:.3f} · P(pay|trial,price)={p_pay_given_trial_price:.3f}"
      f" = {chain:.4f};  direct count: {direct:.4f}")

# 2. Total probability: the overall paid rate is the segments' rates weighted by their shares. The segments must
#    cover everyone: 620 users have no age band, and leaving them out breaks the identity.
by_band = saw.assign(age_band=saw.age_band.fillna("unknown")).groupby("age_band").paid.agg(["mean", "size"])
by_band["P(s)"] = by_band["size"] / by_band["size"].sum()
print(f"Σ P(pay|s)·P(s) = {(by_band['mean'] * by_band['P(s)']).sum():.4f};  P(pay|price) = {saw.paid.mean():.4f}")

# 3. Bayes: P(age band | paid) from P(paid | age band) and P(age band) — and why it is not P(paid | age band).
bayes = by_band["mean"] * by_band["P(s)"] / (by_band["mean"] * by_band["P(s)"]).sum()
direct_b = saw[saw.paid].age_band.fillna("unknown").value_counts(normalize=True)
display(pd.DataFrame({"P(paid | band)": by_band["mean"], "P(band)": by_band["P(s)"],
                      "P(band | paid), Bayes": bayes, "P(band | paid), counted": direct_b}).round(3))

In [ ]:
# 4. Comparing two prices: P(p_3.99 > p_4.99 | data), read off posterior draws.
cmp, rng2 = {}, np.random.default_rng(2)          # its own generator: the later blocks' draws stay the same
for band in band_pop.index:
    a_ = post[(band, "pricing_a")].rvs(20_000, random_state=rng2)
    b_ = post[(band, "pricing_b")].rvs(20_000, random_state=rng2)
    cmp[band] = {"P(3.99 sells better)": (a_ > b_).mean(),
                 "P(4.99 earns more per viewer)": (b_ * 4.99 > a_ * 3.99).mean()}
pd.DataFrame(cmp).T.round(3)

**Read it:** a probability near 0.5 means the data cannot tell the prices apart; near 0 or 1, it can. The second
column asks the business question: revenue per viewer, not conversion. Remember the "given": all of this is
conditional on *already using Ritmo and seeing its premium price*.

**Your notes · Block 2b** — what you think about this task: what was clear, what was hard or confusing, what you would do differently, and your feedback for the teachers. Your own words, a few lines are enough; your teachers read them in your folder.

*Write below this line (double-click this cell, then Shift+Enter):*

…


## Block 3 · Segment the audience by what they do (deck: Part 2, *Segmentation*)

Age bands are a guess about people. Behaviour is evidence. Cluster users on what they did, then look at how the
clusters spread over the age bands (and so over our niches).

In [ ]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score

feat = pd.DataFrame({
    "opens": u.open_app, "habits": u.create_habit,
    "completion": (u.complete_habit / u.create_habit.where(u.create_habit > 0)).fillna(0).clip(0, 3),
    "price_views": u.view_pricing, "trial": (u.start_trial > 0).astype(int), "paid": (u.subscribe > 0).astype(int),
}).fillna(0)
X = StandardScaler().fit_transform(np.log1p(feat))
scores = {k: silhouette_score(X, KMeans(k, n_init=5, random_state=0).fit_predict(X), sample_size=2000, random_state=0) for k in range(3, 7)}
k = max(scores, key=scores.get)
u["segment"] = KMeans(k, n_init=10, random_state=0).fit_predict(X)
print("silhouette by k:", {kk: round(v, 3) for kk, v in scores.items()}, "→ k =", k)
display(feat.groupby(u.segment).mean().round(2).assign(users=u.segment.value_counts()))
pd.crosstab(u.segment, u.age_band, normalize="columns").round(2)

**Your notes · Block 3** — what you think about this task: what was clear, what was hard or confusing, what you would do differently, and your feedback for the teachers. Your own words, a few lines are enough; your teachers read them in your folder.

*Write below this line (double-click this cell, then Shift+Enter):*

…


## Block 3b · Reweighting: your sample is not your market (deck: Part 3)

**Our group mostly prefers reading books to watching series. That tells us almost nothing about Spain.** We are young,
and we all study at university; Spain is not, and education changes reading more than anything else. A sample speaks
for its population only after it is **reweighted** to that population, and only for the parts of the population it
contains at all.

Notation: cells $c$ (here age × education), population counts $N_c$ (Eurostat), sample counts $n_c$, outcome $y$.

**Post-stratification.** $\displaystyle \hat{\bar y}_{\text{PS}} = \sum_c \frac{N_c}{N}\,\bar y_c$, that is weights
$\displaystyle w_i = \frac{N_{c(i)}/N}{n_{c(i)}/n}$.

**Raking** (iterative proportional fitting): when only the **margins** are known (age, education separately), repeat
over margins $m$ and their categories $a$: $\displaystyle w_i \leftarrow w_i \,\frac{N_{m,a}}{\sum_{j:\,m(j)=a} w_j}$ until they agree.

**Inverse probability weighting.** $\displaystyle w_i = \frac{1}{\hat\pi(x_i)}$, with $\hat\pi(x) = P(\text{in the sample} \mid x)$
estimated against a reference (the population, or a probability sample).

**Calibration** (Deville & Särndal, 1992): the weights closest to the design weights $d_i$ that reproduce known totals,
$\displaystyle \min_w \sum_i G(w_i, d_i)$ subject to $\displaystyle \sum_i w_i x_i = X$; raking and post-stratification are special cases.

**How far can you reweight?**
* **Positivity**: a cell with $N_c > 0$ and $n_c = 0$ cannot be reached by any weight. The population share in such
  cells is what your sample cannot speak for.
* **Kish's effective sample size** $\displaystyle n_{\text{eff}} = \frac{\left(\sum_i w_i\right)^2}{\sum_i w_i^2}$, and the design
  effect $\text{deff} = n / n_{\text{eff}} \approx 1 + \text{CV}^2(w)$: unequal weights cost precision.
* **Extreme weights**: one person standing for thousands; trim (cap at $k \times$ median, renormalise) and report both.
* **Distance before and after**: the total-variation distance between sample and population composition,
  $\text{TVD} = \tfrac12 \sum_c \left| \tfrac{n_c}{n} - \tfrac{N_c}{N} \right|$.
* **Representativeness (R-indicator)**: $R = 1 - 2\,S(\hat\rho)$, where $S$ is the standard deviation of the estimated
  inclusion propensities (Schouten, Cobben & Bethlehem, 2009): 1 when everyone was equally likely to be in the sample.

In [ ]:
import itertools
def eurostat_frame(d: dict) -> pd.DataFrame:
    """Any Eurostat JSON-stat answer as a long table (one row per value)."""
    ids, size = d["id"], d["size"]
    dims = {k: (d["dimension"][k]["category"] if "category" in d["dimension"][k] else d["dimension"][k]) for k in ids}
    rows = []
    for combo in itertools.product(*[sorted(dims[k]["index"].items(), key=lambda kv: kv[1]) for k in ids]):
        flat = 0
        for (code, pos), n_ in zip(combo, size):
            flat = flat * n_ + pos
        v = d["value"].get(str(flat))
        if v is not None:
            rows.append({**{k: code for k, (code, _) in zip(ids, combo)}, "value": v})
    return pd.DataFrame(rows)

RW_QUERIES = {
    "books_es_2022": "ilc_scp27?format=JSON&geo=ES&lang=EN&sex=T&time=2022&n_book=0"
                     + "".join(f"&age={a}" for a in ("Y16-24", "Y25-34", "Y35-44", "Y45-54", "Y55-64", "Y65-74", "Y_GE16"))
                     + "".join(f"&isced11={i}" for i in ("TOTAL", "ED0-2", "ED3_4", "ED5-8")),
    "population_edu_es_2024": "lfsa_pgaed?format=JSON&geo=ES&lang=EN&sex=T&time=2024"
                     + "".join(f"&age={a}" for a in ("Y15-19", "Y20-24", "Y25-29", "Y30-34", "Y35-39", "Y40-44", "Y45-49",
                                                     "Y50-54", "Y55-59", "Y60-64", "Y65-69"))
                     + "".join(f"&isced11={i}" for i in ("ED0-2", "ED3_4", "ED5-8")),
    "vod_es_2024": "isoc_ci_ac_i?format=JSON&geo=ES&lang=EN&time=2024&unit=PC_IND&indic_is=I_IUVOD"
                     + "".join(f"&ind_type={a}" for a in ("IND_TOTAL", "Y16_24LO", "Y16_24ME", "Y16_24HI", "Y25_54LO",
                                                          "Y25_54ME", "Y25_54HI", "Y55_74LO", "Y55_74ME", "Y55_74HI")),
}
rw_src = {}
def eurostat_load(key: str) -> pd.DataFrame:
    try:
        d = json.load(urllib.request.urlopen(EUROSTAT + RW_QUERIES[key], timeout=20)); rw_src[key] = "Eurostat live " + TODAY
    except Exception as e:
        d = json.loads((EX / "eurostat_snapshot_2026-10-29.json").read_text())[key]; rw_src[key] = f"Eurostat snapshot 2026-10-29 ({type(e).__name__})"
    return eurostat_frame(d)

books, popedu, vod = (eurostat_load(k) for k in RW_QUERIES)
AGE6, EDU3 = ["16-24", "25-34", "35-44", "45-54", "55-64", "65-69"], ["ED0-2", "ED3_4", "ED5-8"]
P5 = popedu.set_index(["age", "isced11"]).value * 1000
FIVE = {"16-24": [("Y15-19", 0.8), ("Y20-24", 1)], "25-34": [("Y25-29", 1), ("Y30-34", 1)], "35-44": [("Y35-39", 1), ("Y40-44", 1)],
        "45-54": [("Y45-49", 1), ("Y50-54", 1)], "55-64": [("Y55-59", 1), ("Y60-64", 1)], "65-69": [("Y65-69", 1)]}
BOOK_AGE = {"16-24": "Y16-24", "25-34": "Y25-34", "35-44": "Y35-44", "45-54": "Y45-54", "55-64": "Y55-64", "65-69": "Y65-74"}
VOD_AGE = {"16-24": "Y16_24", "25-34": "Y25_54", "35-44": "Y25_54", "45-54": "Y25_54", "55-64": "Y55_74", "65-69": "Y55_74"}
EDU_CODE = {"ED0-2": "LO", "ED3_4": "ME", "ED5-8": "HI"}
nob = books.set_index(["age", "isced11"]).value
vodv = vod.set_index("ind_type").value
cells = pd.DataFrame([{"age": a, "edu": e, "N": sum(P5[(c, e)] * k for c, k in FIVE[a]),
                       "reads_book": 1 - nob[(BOOK_AGE[a], e)] / 100, "watches_vod": vodv[VOD_AGE[a] + EDU_CODE[e]] / 100}
                      for a in AGE6 for e in EDU3]).set_index(["age", "edu"])
cells["share"] = cells.N / cells.N.sum()
spain_cells = (cells[["reads_book", "watches_vod"]].mul(cells.share, axis=0)).sum()
print(rw_src)
print(f"Spain from 18 cells (16–69): reads a book {spain_cells.reads_book:.3f}, watches VOD {spain_cells.watches_vod:.3f}")
print(f"Spain as published:         reads a book {1 - nob[('Y_GE16', 'TOTAL')] / 100:.3f} (16+, 2022), watches VOD {vodv['IND_TOTAL'] / 100:.3f} (16–74, 2024)")
cells.round(3)

**Read it:** rebuilding Spain from its 18 cells gives nearly the published totals: the cells are a faithful model of
the population, so weighting *to* them is meaningful. Notice the rows: in every age group, people with tertiary
education (`ED5-8`) read far more. A university class will read more than Spain for that reason alone.

In [ ]:
# Our group: one row per person. DEMO answers until your class answers (a show of hands or the session's poll):
# change the counts below, keep the columns. Ask the questions the way Eurostat does, so they mean the same thing:
# "read at least one book in the last 12 months?" and "watched films or series on a paid streaming service lately?"
CLASS = pd.DataFrame(
    [("16-24", "ED3_4", 1, 0)] * 6 + [("16-24", "ED3_4", 1, 1)] + [("16-24", "ED3_4", 0, 1)] +
    [("25-34", "ED5-8", 1, 0)] * 6 + [("25-34", "ED5-8", 1, 1)] * 3 + [("25-34", "ED5-8", 0, 1)] * 4,
    columns=["age", "edu", "reads_book", "watches_vod"])
raw = CLASS[["reads_book", "watches_vod"]].mean()
print(f"our group, raw (n={len(CLASS)}): reads a book {raw.reads_book:.2f}, watches VOD {raw.watches_vod:.2f}"
      f"  → {'books' if raw.reads_book > raw.watches_vod else 'series'} win")
print(f"Spain (Eurostat):              reads a book {spain_cells.reads_book:.2f}, watches VOD {spain_cells.watches_vod:.2f}"
      f"  → {'books' if spain_cells.reads_book > spain_cells.watches_vod else 'series'} win")

def reweight_report(w: np.ndarray, cell: pd.Series, target: pd.Series) -> dict:
    """How far this reweighting can be trusted: coverage, effective n, design effect, extreme weights, distance."""
    w = np.asarray(w, float)
    n_eff = w.sum() ** 2 / (w ** 2).sum()
    samp = cell.value_counts(normalize=True).reindex(target.index, fill_value=0)
    after = pd.Series(w, index=cell.values).groupby(level=0).sum().reindex(target.index, fill_value=0) / w.sum()
    tgt = target / target.sum()
    uncovered = tgt[samp == 0].sum()
    out = {"n": len(w), "n_eff": round(float(n_eff), 1), "deff": round(float(len(w) / n_eff), 2),
           "max/median weight": round(float(w.max() / np.median(w)), 1),
           "TVD before": round(float(0.5 * (samp - tgt).abs().sum()), 3), "TVD after": round(float(0.5 * (after - tgt).abs().sum()), 3),
           "population share the sample cannot reach": round(float(uncovered), 3)}
    out["verdict"] = ("cannot speak for this population: restrict the target or collect the missing cells" if uncovered > 0.05
                      else "too few effective answers" if n_eff < 30 else "usable, with the n_eff above")
    return out

In [ ]:
# 1. Post-stratify our group to the WHOLE adult population (16–69): most of Spain is in cells we do not have.
cell_id = CLASS.age + "|" + CLASS.edu
target_all = pd.Series(cells.N.values, index=[f"{a}|{e}" for a, e in cells.index])
w_all = cell_id.map((target_all / target_all.sum()) / cell_id.value_counts(normalize=True))
print("to all of Spain 16–69:", reweight_report(w_all, cell_id, target_all))

# 2. Restrict the target to the cells we cover ("Spaniards like us"): now reweighting is legitimate.
covered = target_all[target_all.index.isin(cell_id.unique())]
w_cov = cell_id.map((covered / covered.sum()) / cell_id.value_counts(normalize=True))
ps = CLASS[["reads_book", "watches_vod"]].mul(w_cov, axis=0).sum() / w_cov.sum()
truth_cov = cells.loc[[tuple(i.split("|")) for i in covered.index], ["reads_book", "watches_vod"]].mul(
    (covered / covered.sum()).values, axis=0).sum()
print("to the covered cells:", reweight_report(w_cov, cell_id, covered))
pd.DataFrame({"our group, raw": raw, "our group, post-stratified to people like us": ps,
              "Eurostat, people like us": truth_cov, "Eurostat, all of Spain 16–69": spain_cells}).T.round(3)

**Read it:** reweighting fixed the *mix* inside the cells we have (bachelor-age and master-age students in their
true proportions), and our group still reads more than Eurostat's people like us: that difference is **us**, not our
composition, and no weight removes it. For all of Spain, the honest statement is a **bound**: with the uncovered share
$u$ unknown, $\;(1-u)\,\hat y_{\text{covered}} \le \bar y_{\text{Spain}} \le (1-u)\,\hat y_{\text{covered}} + u$ (Manski's
worst-case bounds, the same idea as the Fréchet bounds of block 8b). A **model** can go further (for example: assume our
deviation from Eurostat is the same in every cell), but that is an assumption, and the ledger says so.

In [ ]:
u_share = 1 - covered.sum() / target_all.sum()
bounds = pd.DataFrame({"low": (1 - u_share) * ps, "high": (1 - u_share) * ps + u_share})
ratio = (ps / truth_cov)                                         # "our deviation", assumed to hold in every cell
model = cells[["reads_book", "watches_vod"]].mul(ratio, axis=1).clip(0, 1).mul(cells.share, axis=0).sum()
print(f"uncovered share of Spain 16–69: {u_share:.2f}")
pd.concat({"worst-case bounds": bounds.T, "model (an assumption)": model.to_frame().T.rename(index={0: "estimate"})}).round(3)

In [ ]:
# 3. A real sample, microdata: Ritmo's users in Spain, reweighted to Spain's adults by age (block 1's Eurostat counts).
es = u[(u.country == "ES") & u.age_band.notna()].copy()
es["paid"] = es.subscribe > 0
target_age = band_pop.copy()
es["w_ps"] = es.age_band.map((target_age / target_age.sum()) / es.age_band.value_counts(normalize=True))
print("Ritmo ES → Spain by age:", reweight_report(es.w_ps, es.age_band, target_age))

# Inverse probability weighting gives the same weights when x is one categorical variable: a check of the method.
from sklearn.linear_model import LogisticRegression
Xs = pd.get_dummies(es.age_band).astype(float)
Xp = pd.get_dummies(pd.Series(target_age.index)).astype(float)
X_ = pd.concat([Xs, Xp]); y_ = np.r_[np.ones(len(Xs)), np.zeros(len(Xp))]
sw = np.r_[np.ones(len(Xs)), target_age.values / target_age.sum() * len(Xs)]   # the population, scaled to the sample's size
pi = LogisticRegression(C=1e6, max_iter=1000).fit(X_, y_, sample_weight=sw).predict_proba(Xs)[:, 1]
es["w_ipw"] = (1 - pi) / pi
print("IPW vs post-stratification, correlation of weights:", round(np.corrcoef(es.w_ipw, es.w_ps)[0, 1], 4))

# Raking on two margins, for the phone users (web users have no phone system): age (open data) and the phone's
# system (an assumption here: in your project, put a source). Raking needs only the margins, not their joint table.
IOS = fact("ios_share_es", 0.2, 0.3, "share of smartphones in Spain on iOS", "assumption", "to check: an OS market-share source")
ph = es[es.device.isin(["ios", "android"])].copy()
margins = {"age_band": target_age / target_age.sum(), "device": pd.Series({"ios": np.mean(IOS), "android": 1 - np.mean(IOS)})}
w = np.ones(len(ph))
for it in range(100):
    for col, m in margins.items():
        cur = pd.Series(w, index=ph.index).groupby(ph[col]).sum() / w.sum()
        w = w * ph[col].map(m / cur).values
    gap = max((pd.Series(w, index=ph.index).groupby(ph[c]).sum() / w.sum() - m).abs().max() for c, m in margins.items())
    if gap < 1e-8:
        break
ph["w_rake"] = w
w_t = ph.w_rake.values.copy()                                     # trim: cap at 5 × the median, renormalise, repeat
for _ in range(100):                                              # (renormalising lifts the median, so one pass is not enough)
    cap = 5 * np.median(w_t)
    if w_t.max() <= cap * (1 + 1e-9):
        break
    w_t = np.minimum(w_t, cap) * ph.w_rake.sum() / np.minimum(w_t, cap).sum()
ph["w_trim"] = w_t
print(f"raking: {it + 1} rounds, largest margin gap {gap:.1e};", "phone users:", len(ph), "of", len(es))
def est(d, col):
    w_ = np.ones(len(d)) if col == "raw" else d[col].values
    return {"paid share": float(np.average(d.paid, weights=w_)), "n_eff": float(w_.sum() ** 2 / (w_ ** 2).sum()),
            "max/median": float(w_.max() / np.median(w_))}
pd.DataFrame({"raw (all ES users)": est(es, "raw"), "post-stratified by age": est(es, "w_ps"), "IPW by age": est(es, "w_ipw"),
              "raw (phone users)": est(ph, "raw"), "raked: age × system": est(ph, "w_rake"), "raked, trimmed at 5× median": est(ph, "w_trim")}).T.round(3)

**Read it:** the weights move Ritmo's Spanish users toward Spain's age mix; the estimate moves a little, and the
effective sample shrinks. The ledger records the reweighted paid share, with the weighting it used, as `partial`
evidence. The rule for your project: **reweight what you can cover, say what you cannot, and report $n_{\text{eff}}$
with every weighted number.**

**Your notes · Block 3b** — what you think about this task: what was clear, what was hard or confusing, what you would do differently, and your feedback for the teachers. Your own words, a few lines are enough; your teachers read them in your folder.

*Write below this line (double-click this cell, then Shift+Enter):*

…


## Block 4 · Simulated users: shape, never level (deck: Part 4)

Two kinds of simulation:

1. **Agent-based, offline** (below): each simulated person has a price they would pay, drawn per niche; the draws are
   *calibrated* so that at Ritmo's prices they reproduce block 2's observed rates. Then we ask the simulation a
   question the data cannot answer: what happens at 7.99 for a coach?
2. **LLM personas** (optional, needs a key): a model answers as a persona of each niche, on a 1–5 purchase-intent scale.
   The research (deck, Part 4) says such answers can rank options well and get levels wrong, and are less varied than
   real people. So the ledger records them as `simulated`, and block 8 uses them only for **relative** differences.

In [ ]:
def calibrated_wtp(band: str, n: int = 20_000) -> np.ndarray:
    """Lognormal willingness to pay whose P(WTP ≥ 3.99) and P(WTP ≥ 4.99) match block 2's medians for this band."""
    from scipy.optimize import brentq
    p399, p499 = (post[(band, a)].median() for a in ("pricing_a", "pricing_b"))
    p499 = min(p499, p399 * 0.98)                      # a higher price never sells more
    z1, z2 = stats.norm.isf(p399), stats.norm.isf(p499)
    sigma = max((np.log(4.99) - np.log(3.99)) / max(z2 - z1, 1e-3), 0.3)
    mu = np.log(3.99) - sigma * z1
    return rng.lognormal(mu, sigma, n)

sim = pd.DataFrame({b: [ (calibrated_wtp(b) >= p).mean() for p in (3.99, 4.99, 7.99, 9.99)] for b in band_pop.index},
                   index=["3.99", "4.99", "7.99", "9.99"]).T
for b in sim.index:
    fact(f"sim_share_799_{b}", sim.loc[b, "7.99"] * 0.5, sim.loc[b, "7.99"] * 1.5, "share paying ≥7.99 (of viewers)", "simulated",
         "lognormal WTP calibrated on Ritmo's 3.99/4.99 test", "shape only")
sim.round(3)

In [ ]:
# Optional: LLM personas. Put ANTHROPIC_API_KEY in Colab's secrets (key icon) or skip this cell.
PERSONA_PROMPT = (EX / "persona_simulator.md").read_text() if (EX / "persona_simulator.md").exists() else "Answer as the persona. Reply with JSON {intent: 1-5, reason: str}."
def ask_persona(niche: str, price: float, model: str = "claude-haiku-5-5") -> dict | None:
    import os
    try:
        from google.colab import userdata; key = userdata.get("ANTHROPIC_API_KEY")
    except Exception:
        key = os.environ.get("ANTHROPIC_API_KEY")
    if not key:
        return None
    import anthropic
    msg = anthropic.Anthropic(api_key=key).messages.create(model=model, max_tokens=200, system=PERSONA_PROMPT,
        messages=[{"role": "user", "content": f"You are in the niche '{niche}': {NICHES.loc[niche, 'job']}. "
                   f"Ritmo Coach costs {price} EUR a month. How likely are you to buy it?"}])
    return json.loads(re.search(r"\{.*\}", msg.content[0].text, re.S).group(0))

print(ask_persona("students", 4.99) or "no key: skipped (the offline simulation above is enough for today)")

**Your notes · Block 4** — what you think about this task: what was clear, what was hard or confusing, what you would do differently, and your feedback for the teachers. Your own words, a few lines are enough; your teachers read them in your folder.

*Write below this line (double-click this cell, then Shift+Enter):*

…


## Block 5 · Competitors, segmented (deck: Part 2, *Competitors*)

`competitors.csv` was filled by a research pass on 29 Oct 2026 (sources in its `source` column; check them: prices
change). We place each product on what it charges and how much coaching it does, cluster them, and look for the
**empty corner**: a niche no one serves at a price people pay.

In [ ]:
import io
COMPETITORS_CSV = """product,price_month_eur,model,coaching,social,ai,audience,platforms,source,checked
Habitica,4.00,freemium; $47.99/yr (≈ $ per month shown; $≈€),1,1,0,gamers and students,web ios android,https://apps.apple.com/app/id994882113,2026-10-29 (store listing via search)
Streaks,0.42,one-off purchase $4.99–5.99 (shown over 12 months; UNVERIFIED),0,0,0,Apple users,ios,https://toolradar.com/tools/streaks/pricing,2026-10-29 (secondary)
Fabulous,3.75,subscription $39.99–49.99/yr with heavy price tests,2,0,0,routine and wellbeing seekers,ios android,https://apps.apple.com/US/app/id1203637303,2026-10-29 (store listing via search)
Finch,5.83,free core; Finch Plus $9.99/mo or $69.99/yr,1,1,0,young adults; mental self-care,ios android,https://apppricinglab.com/iap/apple/1528595748,2026-10-29 (secondary)
Habitify,3.33,Free/Plus/Pro since 6 Apr 2026; $7.49/mo or $39.99/yr,0,0,0,productivity; cross-platform,ios android web,https://feedback.habitify.me/changelog/new-tiered-pricing,2026-10-29
Atoms,3.33,freemium; Pro $4.99/mo or $39.99/yr,1,0,0,Atomic Habits readers,ios android,https://apppricinglab.com/iap/apple/6474421906,2026-10-29 (secondary; 30 May 2026)
Way of Life,2.50,free + premium $29.99/yr,0,0,0,quantified-self users,ios android,https://apppricinglab.com/iap/apple/393159800,2026-10-29 (secondary; Jul 2026)
Duolingo Super (reference),7.00,≈$83.99/yr in the US; varies by test (UNVERIFIED),2,1,1,mass market; streak-driven,ios android web,https://www.mindomax.com/duolingo-cost,2026-10-29 (secondary)
Ritmo premium,4.49,A/B test 3.99 vs 4.99 per month,0,0,0,habit builders,ios android,course data events.csv,2026-10-29
"""   # the copy of 29 Oct 2026, if examples/session-09 is not reachable
comp = pd.read_csv(EX / "competitors.csv") if (EX / "competitors.csv").exists() else pd.read_csv(io.StringIO(COMPETITORS_CSV))
display(comp)
cf = comp[["price_month_eur", "coaching", "social", "ai"]].astype(float)
comp["cluster"] = KMeans(3, n_init=10, random_state=0).fit_predict(StandardScaler().fit_transform(cf))
import matplotlib.pyplot as plt
ax = comp.plot.scatter("coaching", "price_month_eur", c="cluster", cmap="viridis", s=60, figsize=(6, 4), colorbar=False)
for _, r in comp.iterrows():
    ax.annotate(r["product"], (r.coaching, r.price_month_eur), fontsize=8, xytext=(3, 3), textcoords="offset points")
ax.scatter([3], [4.99], marker="*", s=250, c="red"); ax.annotate("Ritmo Coach?", (3, 4.99), color="red", xytext=(5, -12), textcoords="offset points")
ax.set_xlabel("coaching depth (0 none … 3 personal plan)"); ax.set_ylabel("€ / month"); plt.show()

**Your notes · Block 5** — what you think about this task: what was clear, what was hard or confusing, what you would do differently, and your feedback for the teachers. Your own words, a few lines are enough; your teachers read them in your folder.

*Write below this line (double-click this cell, then Shift+Enter):*

…


## Block 5b · Features from existing apps (deck: Part 2, *Features from existing apps*)

What do the products people already use **do**, and what do their users **say** about each feature? Two public
sources from Apple's App Store (no key): each app's description (what it claims) and its 500 most recent US reviews
(what users report). Snapshot of 29 Oct 2026: 8 apps, 3,999 reviews (`app_descriptions.csv`, `app_reviews.csv.gz`;
reviewers' names were never stored).

Extraction, two ways: a **lexicon** (regular expressions, below: fast, transparent, misses paraphrases) and an **LLM**
with `feature_extraction.md` (optional cell: reads paraphrases, must be checked against the lexicon on a sample).

In [ ]:
apps = pd.read_csv(EX / "app_descriptions.csv") if (EX / "app_descriptions.csv").exists() else None
reviews = pd.read_csv(EX / "app_reviews.csv.gz") if (EX / "app_reviews.csv.gz").exists() else None
if reviews is None:   # on Colab without the examples folder: the same public feeds, live (about a minute)
    import time as _time
    def _get(u):
        try:
            r = json.load(urllib.request.urlopen(urllib.request.Request(u, headers={"User-Agent": "Mozilla/5.0"}), timeout=20))
        except Exception:
            r = None
        _time.sleep(3)                                            # Apple allows about 20 calls a minute
        return r
    IDS = {"Habitica": 994882113, "Fabulous": 1203637303, "Finch": 1528595748, "Atoms": 6474421906, "Way of Life": 393159800,
           "Habitify": 1111447047, "Streaks": 963034692, "Productive": 983826477}
    rows_a, rows_r = [], []
    for n, i in IDS.items():
        found = (_get(f"https://itunes.apple.com/lookup?id={i}&country=us") or {}).get("results") or [{}]
        rows_a.append({"app": n, "description": found[0].get("description", "")})
        feed = (_get(f"https://itunes.apple.com/us/rss/customerreviews/page=1/id={i}/sortby=mostrecent/json") or {}).get("feed", {})
        entries = feed.get("entry", []) if isinstance(feed, dict) else []
        rows_r += [{"app": n, "rating": int(e["im:rating"]["label"]), "title": e["title"]["label"], "text": e["content"]["label"]}
                   for e in (entries if isinstance(entries, list) else [entries]) if isinstance(e, dict) and "im:rating" in e]
    apps = pd.DataFrame(rows_a, columns=["app", "description"])
    reviews = pd.DataFrame(rows_r, columns=["app", "rating", "title", "text"])
    print(f"App Store, live: {reviews.app.nunique()} apps, {len(reviews)} reviews"
          + ("" if reviews.app.nunique() == len(IDS) else " — Apple's feed limits calls and sometimes returns none: run this cell "
             "again later, or add the course folder to your Drive, where examples/session-09 holds the snapshot"))

FEATURES = {
    "reminders":            r"\b(?:remind\w*|notification\w*|alerts?|nudg\w*)\b",
    "streaks":              r"\bstreaks?\b",
    "gamification":         r"\b(?:game\w*|rpg|level(?:s|ing)? up|points|rewards?|pet|avatar|quests?)\b",
    "social_accountability":r"\b(?:friends?|party|guild|accountab\w*|together|share|social|community)\b",
    "coaching_guidance":    r"\b(?:coach\w*|guid\w*|journey|programs?|plans?|lessons?|teach\w*|tips|advice)\b",
    "stats_progress":       r"\b(?:stats?|statistic\w*|charts?|graphs?|progress|analytics|history|calendar)\b",
    "flexible_schedule":    r"\b(?:flexib\w*|schedul\w*|skip\w*|days? off|vacation|weekly|times? per week|custom\w*)\b",
    "widgets_watch":        r"\b(?:widgets?|apple watch|watch app|shortcuts?|siri)\b",
    "sync_backup":          r"\b(?:sync\w*|icloud|backup|lost (?:my )?data|cross[- ]platform|android)\b",
    "price_subscription":   r"\b(?:subscri\w*|pay\w*|price\w*|expensive|premium|paywall|free trial|charged?|refund)\b",
    "mood_selfcare":        r"\b(?:mood|self[- ]care|anxiety|depress\w*|mental health|journal\w*|reflect\w*)\b",
    "ai":                   r"\b(?:ai|a\.i\.|chatgpt|artificial intelligence|smart suggestions?)\b",
}
claims = pd.DataFrame({f: apps.description.fillna("").str.lower().str.contains(p) for f, p in FEATURES.items()}).set_index(apps.app).astype(int).T
rtext = (reviews.title.fillna("") + " " + reviews.text.fillna("")).str.lower()
F = pd.DataFrame({f: rtext.str.contains(p) for f, p in FEATURES.items()})
summary = pd.DataFrame({"apps_claiming": claims.sum(axis=1), "reviews_mentioning": F.sum(),
                        "rating_gap": {f: reviews.rating[F[f]].mean() - reviews.rating.mean() for f in F}})
print(len(reviews), "reviews of", reviews.app.nunique(), "apps; mean rating", round(reviews.rating.mean(), 2))
summary.sort_values("rating_gap").round(2)

**Read it:** `apps_claiming` is what the market offers; `reviews_mentioning` is what users talk about; `rating_gap` is
how a review that mentions the feature rates against the average. A strongly negative gap usually means the feature
**fails or is missing** where users expect it (price, sync); a positive gap, that it **delights** (social, mood).
Watch the `ai` row: few reviews, and the ones that mention AI rate far lower. A product called "AI coach" walks into
that.

In [ ]:
# Optional: LLM extraction with feature_extraction.md, checked against the lexicon on a sample (agreement per feature).
def llm_features(texts: list[str], model: str = "claude-haiku-5-5") -> list[dict] | None:
    import os
    try:
        from google.colab import userdata; key = userdata.get("ANTHROPIC_API_KEY")
    except Exception:
        key = os.environ.get("ANTHROPIC_API_KEY")
    if not key:
        return None
    import anthropic
    brief = (EX / "feature_extraction.md").read_text() if (EX / "feature_extraction.md").exists() else "Return JSON {features: [..]}"
    client, out = anthropic.Anthropic(api_key=key), []
    for t in texts:
        m = client.messages.create(model=model, max_tokens=300, system=brief,
                                   messages=[{"role": "user", "content": f"Features: {list(FEATURES)}\nReview: {t}"}])
        out.append(json.loads(re.search(r"\{.*\}", m.content[0].text, re.S).group(0)))
    return out

sample = reviews.sample(min(30, len(reviews)), random_state=1)
got = llm_features((sample.title + " " + sample.text).tolist()) if len(sample) else None
if got is None:
    print("no key: skipped (the lexicon above is the extraction today)")
else:
    llm = pd.DataFrame([{f: f in g.get("features", []) for f in FEATURES} for g in got], index=sample.index)
    print("agreement with the lexicon, per feature:"); display((llm == F.loc[sample.index]).mean().round(2))

**Your notes · Block 5b** — what you think about this task: what was clear, what was hard or confusing, what you would do differently, and your feedback for the teachers. Your own words, a few lines are enough; your teachers read them in your folder.

*Write below this line (double-click this cell, then Shift+Enter):*

…


## Block 6 · What people say: Hacker News, live (deck: Part 2, *Conversations*)

Public conversations are the cheapest evidence of a **job** and its pains, and the least representative (who posts on
Hacker News is not who buys a habit app). We read comments that mention habit trackers since 2023 through HN's open
Algolia API (fallback: the snapshot `hn_habit_comments.csv.gz`), find topics, and count how often each niche's cue
words appear next to a pain.

In [ ]:
def hn_comments(queries=("habit tracker", "habit app", "habit tracking", "streaks app", "procrastination app"), pages=2):
    rows = []
    for q in queries:
        for page in range(pages):
            url = "https://hn.algolia.com/api/v1/search?" + urllib.parse.urlencode(
                {"query": q, "tags": "comment", "hitsPerPage": 100, "page": page, "numericFilters": "created_at_i>1672531200"})
            for h in json.load(urllib.request.urlopen(url, timeout=20))["hits"]:
                t = re.sub(r"\s+", " ", re.sub(r"<[^>]+>", " ", h.get("comment_text") or "")).strip()
                rows.append(dict(comment_id=h["objectID"], created_at=h["created_at"][:10], query=q, text=t))
    return pd.DataFrame(rows).drop_duplicates("comment_id")
try:
    hn = hn_comments(); hn_src = "HN Algolia API, live " + TODAY
except Exception as e:
    hn = pd.read_csv(EX / "hn_habit_comments.csv.gz"); hn_src = f"HN snapshot 2026-10-29 ({type(e).__name__})"
print(hn_src, len(hn), "comments")

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import NMF
tf = TfidfVectorizer(max_df=0.5, min_df=5, stop_words="english", ngram_range=(1, 2))
W = tf.fit_transform(hn.text.fillna(""))
nmf = NMF(8, random_state=0).fit(W)
terms = np.array(tf.get_feature_names_out())
pd.DataFrame({f"topic {i}": terms[c.argsort()[-8:][::-1]] for i, c in enumerate(nmf.components_)}).T

In [ ]:
CUES = {"students": r"\b(student|exam|study|studying|university|college|thesis)\b",
        "remote_workers": r"\b(remote|work from home|wfh|deep work|meetings?|focus)\b",
        "parents": r"\b(kids?|children|parent|family|toddler)\b",
        "health_55": r"\b(doctor|blood pressure|medication|retired|walking|weight)\b"}
PAIN = r"\b(stopp?ed|quit|gave up|forget|forgot|guilt|streak|broke|abandon|nag|annoying|too many|subscription|expensive)\b"
t = hn.text.fillna("").str.lower()
voice = pd.DataFrame({n: {"mentions": t.str.contains(c).sum(), "with_pain": (t.str.contains(c) & t.str.contains(PAIN)).sum()}
                      for n, c in CUES.items()}).T
voice["share_of_voice"] = voice.mentions / voice.mentions.sum()
for n, r in voice.iterrows():
    fact(f"voice_{n}", r.share_of_voice * 0.7, r.share_of_voice * 1.3, "share of niche mentions", "research", hn_src, "HN is not the buyers: shape only")
voice

**Your notes · Block 6** — what you think about this task: what was clear, what was hard or confusing, what you would do differently, and your feedback for the teachers. Your own words, a few lines are enough; your teachers read them in your folder.

*Write below this line (double-click this cell, then Shift+Enter):*

…


## Block 6b · Which features matter to which user group (deck: Part 2, *Which features matter to whom*)

A review that says "my kids", "my exams" or "my doctor" tells us who wrote it, roughly. Within each group we measure,
for each feature $f$:

* **salience** $P(f \mid g)$: the share of the group's reviews that mention it, with a Beta interval (groups are small);
* **lift** $P(f \mid g) / P(f)$: how much more this group talks about it than everyone (Bayes in disguise:
  $\frac{P(f \mid g)}{P(f)} = \frac{P(g \mid f)}{P(g)}$);
* **valence**: the rating gap of the group's reviews that mention it.

Then a rough **Kano** reading: a feature with negative valence is a *must-be* (people punish its absence), one with
positive valence a *delighter*, the rest *performance*.

In [ ]:
GROUP_CUES = {
    "students":       r"\b(?:student|school|college|universit\w*|exams?|study|studying|homework|class(?:es)?)\b",
    "remote_workers": r"\b(?:work(?:ing)? from home|remote|job|office|meetings?|career|boss|work ?day)\b",
    "parents":        r"\b(?:kids?|children|son|daughter|mom|mum|dad|parent\w*|family|toddler|baby)\b",
    "health_55":      r"\b(?:doctor|retired|retirement|my age|older|senior|blood pressure|medication|diabet\w*|grandkids?)\b",
}
G = pd.DataFrame({g: rtext.str.contains(p) for g, p in GROUP_CUES.items()})
rows = []
for g in G:
    n_g = int(G[g].sum())
    for f in F:
        k = int((G[g] & F[f]).sum())
        lo, hi = stats.beta(1 + k, 1 + n_g - k).ppf([0.05, 0.95])
        sal = k / n_g if n_g else np.nan
        val = reviews.rating[G[g] & F[f]].mean() - reviews.rating.mean() if k >= 3 else np.nan
        rows.append(dict(group=g, feature=f, n_group=n_g, mentions=k, salience=sal, sal_lo=lo, sal_hi=hi,
                         lift=sal / F[f].mean(), valence=val))
IMP = pd.DataFrame(rows)
IMP["kano"] = np.select([IMP.valence <= -0.5, IMP.valence >= 0.3], ["must-be", "delighter"], "performance")
IMP.loc[IMP.valence.isna(), "kano"] = "too few"
print("reviews per group:", G.sum().to_dict())
IMP.pivot(index="feature", columns="group", values="lift").round(1)

In [ ]:
# The coach's own features for each group: how much the group cares about guidance, accountability and recovery
# after a slip (flexible schedule) — this weight drives the coach's effect in the journey simulation (block 8c).
COACH_FEATURES = ["coaching_guidance", "social_accountability", "flexible_schedule", "reminders"]
# Reviews that name a group are longer, so they mention every feature more: divide by the group's mean lift over all.
coach_weight = (IMP[IMP.feature.isin(COACH_FEATURES)].groupby("group").lift.mean()
                / IMP.groupby("group").lift.mean()).rename("coach_relevance (relative lift)").clip(0.5, 2.0)
coach_weight = coach_weight.reindex(list(GROUP_CUES)).fillna(1.0)      # a group with no reviews: neutral, not unknown
for g, w in coach_weight.items():
    fact(f"coach_relevance_{g}", w * 0.7, w * 1.3, "relative interest in coach features", "research",
         "App Store reviews (8 apps, 2026-10-29), cue words; small groups", "shape only")
display(coach_weight.round(2))
IMP[IMP.feature.isin(COACH_FEATURES)].pivot(index="feature", columns="group", values="kano")

**Your notes · Block 6b** — what you think about this task: what was clear, what was hard or confusing, what you would do differently, and your feedback for the teachers. Your own words, a few lines are enough; your teachers read them in your folder.

*Write below this line (double-click this cell, then Shift+Enter):*

…


## Block 7 · Automated analysis: an agent writes, the notebook checks (deck: Part 5)

The state of the art (deck, Part 5): agents write analysis code well and choose analyses badly; on realistic
multi-step data tasks the best agents still get a large share wrong, and two runs can disagree. So the pipeline gives
the agent **an instruction file** (`analyst_agent.md`), **a question with a schema**, and keeps the right to **check**:

* **inside the pipeline:** the cell below sends the instruction file and a sample of the data to a model, gets back
  code, runs it here, and accepts the answer only if the notebook's own recomputation agrees;
* **outside the pipeline:** paste `analyst_agent.md` and `audience.csv` into Claude or ChatGPT, ask the same question,
  paste its JSON answer into `outside_answer` and run the same check.

In [ ]:
QUESTION = ("For each age_band: users who saw a price, the share of them who subscribed, and the 90% interval. "
            "Return JSON: {\"rows\": [{\"age_band\": str, \"viewers\": int, \"paid_rate\": float}], \"method\": str}")

def truth() -> pd.DataFrame:
    return saw.groupby("age_band").agg(viewers=("paid", "size"), paid_rate=("paid", "mean"))

def check(answer: dict, tol: float = 0.005) -> pd.DataFrame:
    """The agent's numbers against the notebook's own: every row must match, or the answer is refused."""
    a = pd.DataFrame(answer["rows"]).set_index("age_band")
    c = truth().join(a, rsuffix="_agent")
    c["ok"] = (c.viewers == c.viewers_agent) & ((c.paid_rate - c.paid_rate_agent).abs() <= tol)
    return c

def ask_analyst(model: str = "claude-sonnet-5-5") -> dict | None:
    import os
    try:
        from google.colab import userdata; key = userdata.get("ANTHROPIC_API_KEY")
    except Exception:
        key = os.environ.get("ANTHROPIC_API_KEY")
    if not key:
        return None
    import anthropic
    brief = (EX / "analyst_agent.md").read_text()
    sample = saw.reset_index()[["user_id", "age_band", "arm", "trial", "paid"]].head(30).to_csv(index=False)
    msg = anthropic.Anthropic(api_key=key).messages.create(model=model, max_tokens=1500, system=brief, messages=[{"role": "user",
        "content": f"A DataFrame `saw` has these columns (first 30 rows):\n{sample}\nQuestion: {QUESTION}\n"
                   "Reply with one Python code block that sets a variable `answer` (a dict in that schema)."}])
    code = re.search(r"```(?:python)?\n(.*?)```", msg.content[0].text, re.S).group(1)
    scope = {"saw": saw.copy(), "pd": pd, "np": np, "stats": stats}
    exec(code, scope)                                  # in class: read the code before you run it
    return scope["answer"]

answer = ask_analyst()
outside_answer = None                                  # paste the JSON from a chat here, e.g. {"rows": [...], "method": "..."}
for name, a in (("inside", answer), ("outside", outside_answer)):
    print(name, "→", "skipped" if a is None else ("ACCEPTED" if check(a).ok.all() else "REFUSED"))
    if a is not None:
        display(check(a))

**Your notes · Block 7** — what you think about this task: what was clear, what was hard or confusing, what you would do differently, and your feedback for the teachers. Your own words, a few lines are enough; your teachers read them in your folder.

*Write below this line (double-click this cell, then Shift+Enter):*

…


## Block 8 · Fuse it: a provable niche estimate (deck: Part 6)

**Route A, bottom-up**, per niche, for Spain: what the niche spends a year on an app like ours (its **SAM**) =
`people in the niche's age bands × share who use a habit or self-improvement app × share who look at a price × paid
rate at that price × price × 12`. Our **year-one revenue (SOM)** = SAM × the share we reach.

**Route B, top-down:** a provider's figure for the habit-app market × Spain's share × the niche's share. Providers
disagree (deck, Part 2), so route B is a wide range; its job is to say whether route A is in the right **order of
magnitude**.

The proof checks run before any number is shown: every input is in the ledger, no `simulated` input sets a level, the
two routes' SAMs agree within a factor of 3, and the answer is an interval. A niche that fails a check is not "wrong":
the check says which evidence to distrust (here: Hacker News is not where people over 55 talk).

In [ ]:
ASSUME = {  # said out loud; block 10 replaces the weakest with research
    "habit_app_users_share": fact("habit_app_users_share", 0.04, 0.12, "share of adults using a habit/self-improvement app", "assumption", "to research (block 10)"),
    "reach_share":           fact("reach_share", 0.01, 0.05, "share of the niche we reach in year 1", "assumption", "marketing plan"),
    "see_price_share":       fact("see_price_share", *stats.beta(1 + len(saw), 1 + len(u) - len(saw)).ppf([0.05, 0.95]), "share of app users who open the price screen", "partial", "Ritmo: users with view_pricing / all users"),
    "topdown_market_eur":    fact("topdown_market_eur", 0.5e9, 1.7e9, "world habit-tracking app market, € / year", "research",
                                 "DataHorizzon $0.51B, Wise Guy $1.16B, Straits $1.7B (2024); Econ Market Research $9.93B dropped: above Sensor Tower's $4.5B for all Health & Fitness apps (2025)"),
    "spain_share":           fact("spain_share", 0.015, 0.03, "Spain's share of the world app market", "assumption", "to research"),
}
L = ledger()
SIM_ALLOWED_FOR_LEVEL = False

def draw(name: str, n: int) -> np.ndarray:
    r = L.loc[name]
    assert r.kind != "simulated" or SIM_ALLOWED_FOR_LEVEL, f"{name} is simulated: it may set a shape, not a level"
    return rng.uniform(r.low, r.high, n) if r.high > r.low else np.full(n, r.low)

def route_a(niche: str, price: float = 4.99, arm: str = "pricing_b", n: int = 20_000, extra: dict | None = None) -> pd.DataFrame:
    bands = NICHES.loc[niche, "age_bands"]
    people = sum(draw(f"population_{b}", n) for b in bands) / (len(bands) if niche == "parents" else 1)  # parents: half of two bands
    paid = np.mean([post[(b, arm)].rvs(n, random_state=rng) for b in bands], axis=0)
    out = pd.DataFrame({"people": people, "app_share": draw("habit_app_users_share", n), "reach": draw("reach_share", n),
                        "see_price": draw("see_price_share", n), "paid_rate": paid})
    for k, v in (extra or {}).items():
        out[k] = v(n)
    out["sam_eur"] = out.drop(columns="reach").prod(axis=1) * price * 12
    out["revenue_eur"] = out.sam_eur * out.reach            # year one: the SOM
    return out

def route_b(niche: str, n: int = 20_000) -> np.ndarray:
    share = L.loc[f"voice_{niche}"][["low", "high"]].mean()          # conversation share: shape only, so used as a split
    return draw("topdown_market_eur", n) * draw("spain_share", n) * share

def proof(niche: str, a: pd.DataFrame, b: np.ndarray) -> dict:
    sam_a, sam_b = np.median(a.sam_eur), np.median(b)
    som = np.percentile(a.revenue_eur, [10, 50, 90])
    ratio = sam_a / sam_b
    return {"niche": niche, "SAM_A_p50": sam_a, "SAM_B_p50": sam_b, "A/B": ratio, "routes_agree": 1/3 <= ratio <= 3,
            "SOM_p10": som[0], "SOM_p50": som[1], "SOM_p90": som[2], "inputs_in_ledger": True, "no_simulated_level": True}

res = pd.DataFrame([proof(n, route_a(n), route_b(n)) for n in NICHES.index]).set_index("niche")
res.round(-3).assign(**{"A/B": res["A/B"].round(2)})

In [ ]:
# Which input moves the answer most? Rank correlation of each input with revenue (a tornado, in numbers).
best = res.SOM_p50.idxmax()
a = route_a(best)
inputs = a.drop(columns=["sam_eur", "revenue_eur"]).loc[:, lambda d: d.nunique() > 1]
sens = inputs.corrwith(a.revenue_eur, method="spearman").abs().sort_values(ascending=False)
print("niche:", best); sens.round(2)

**Read it:** the input at the top of this list is where research pays most. If it is an `assumption`, the estimate
is not yet provable *for that input*: block 10 replaces it.

**Your notes · Block 8** — what you think about this task: what was clear, what was hard or confusing, what you would do differently, and your feedback for the teachers. Your own words, a few lines are enough; your teachers read them in your folder.

*Write below this line (double-click this cell, then Shift+Enter):*

…


## Block 8b · Fusion by statistical matching: connect partial views (deck: Part 6, *Data fusion*)

*"Each source sees a different part of the market. Fusion connects those partial views through shared
characteristics, and makes the missing relationships explicit."* We need, per group $x$ (here: age band in Spain),
the share of people who **have the problem** ($A$) **and** **have a relevant budget** ($B$):

$$q(x) = P(A, B \mid X = x) = P(A \mid X = x)\, P(B \mid A, X = x), \qquad \text{qualified audience} = \sum_x N_x\, q(x)$$

* $N_x$: Eurostat population (block 1, **open**).
* $b(x) = P(B \mid x)$: Eurostat, *individuals with a paid subscription to a health or fitness app* (2025, **open**).
* $a(x) = P(A \mid x)$: wants to build a habit and struggles: no open source; an **assumption** to research.
* The overlap: separate sources do not reveal it. Without a bridge, only the **Fréchet bounds** are safe:
  $\max(0,\ a + b - 1) \le q \le \min(a,\ b)$. $q = ab$ **assumes** $A$ and $B$ independent within $x$.
* A **bridge study** measures $A$ and $B$ in the same people. Ritmo's users all have $A$ (they use a habit app):
  their paid share estimates $P(B \mid A, x)$ — for *Ritmo's* $B$, a different definition from Eurostat's (align it!).

Sources: D'Orazio, *Integration of Data Sources in R through Statistical Matching* (uRos 2019) and Istat's R
package `StatMatch`; Conti, Marella & Scanu, "Uncertainty Analysis in Statistical Matching", *Journal of Official
Statistics* 28(1), 2012; D'Orazio, Di Zio & Scanu, *Statistical Matching: Theory and Practice* (Wiley, 2006).

In [ ]:
try:
    sub = json.load(urllib.request.urlopen(EUROSTAT + "isoc_ec_ibgs?format=JSON&geo=ES&lang=EN&unit=PC_IND&time=2025&indic_is=I_BHLFTS2"
                                           + "".join(f"&ind_type={a}" for a in ("Y16_24", "Y25_34", "Y35_44", "Y45_54", "Y55_64", "Y65_74")), timeout=20))
    sub_src = "Eurostat isoc_ec_ibgs I_BHLFTS2, live " + TODAY
except Exception as e:
    sub = json.loads((EX / "eurostat_snapshot_2026-10-29.json").read_text())["app_subscriptions_es"]
    sub_src = f"Eurostat isoc_ec_ibgs I_BHLFTS2, snapshot 2026-10-29 ({type(e).__name__})"
def eurostat_cell(d: dict, **want) -> float:
    dims = d["dimension"]; ids = d["id"]; flat = 0
    for k in ids:
        cat = dims[k]["category"] if "category" in dims[k] else dims[k]
        code = want.get(k) or next(iter(cat["index"]))
        flat = flat * d["size"][ids.index(k)] + cat["index"][code]
    return d["value"].get(str(flat))
EURO_BANDS = {"18-24": ["Y16_24"], "25-34": ["Y25_34"], "35-44": ["Y35_44"], "45-54": ["Y45_54"], "55+": ["Y55_64", "Y65_74"]}
b = pd.Series({band: np.mean([eurostat_cell(sub, ind_type=c, indic_is="I_BHLFTS2", time="2025") for c in codes]) / 100
               for band, codes in EURO_BANDS.items()})
A_RANGE = {"18-24": (0.25, 0.45), "25-34": (0.25, 0.45), "35-44": (0.25, 0.45), "45-54": (0.20, 0.40), "55+": (0.15, 0.35)}
ritmo_paid = u.assign(paid=u.subscribe > 0).groupby("age_band").paid.mean()          # the bridge: all of them have A
match = []
for band in band_pop.index:
    fact(f"budget_{band}", b[band] * 0.9, b[band] * 1.1, "share with a paid health/fitness app subscription", "open", sub_src)
    fact(f"problem_{band}", *A_RANGE[band], "share who want a habit and struggle", "assumption", "to research: a survey or the bridge")
    a_mid = np.mean(A_RANGE[band])
    match.append(dict(band=band, N=band_pop[band], a=a_mid, b=b[band],
                      q_low=max(0, a_mid + b[band] - 1), q_indep=a_mid * b[band], q_high=min(a_mid, b[band]),
                      q_bridge=a_mid * ritmo_paid[band]))
M = pd.DataFrame(match).set_index("band")
for c in ("q_low", "q_indep", "q_high", "q_bridge"):
    M["aud_" + c[2:]] = M.N * M[c]
M["bridge_within_bounds"] = (M.q_bridge >= M.q_low) & (M.q_bridge <= M.q_high)
M.round(3)

**Read it:** the bounds are wide (from 0 to the smaller rate): that is the honest answer when nobody measured the
overlap. Independence (`q_indep`) is a single number that only *looks* precise. Ritmo's bridge lands inside the bounds
for four bands, nearer the top than independence (people with the problem pay more often), and **outside** them for
55+: there it claims more payers than Eurostat says exist. A real overlap cannot exceed $\min(a, b)$, so the bridge's
$B$ (paying Ritmo, among Ritmo's own users) is not Eurostat's $B$ for that group: align the definitions and the
population (step 1 of the infographic) before you fuse. The fix is a small **bridge study** of real people in Spain
asking both questions: 200 answers per group buy more than another large disconnected dataset.

**Your notes · Block 8b** — what you think about this task: what was clear, what was hard or confusing, what you would do differently, and your feedback for the teachers. Your own words, a few lines are enough; your teachers read them in your folder.

*Write below this line (double-click this cell, then Shift+Enter):*

…


## Block 8c · The key user journey, simulated per group (deck: Part 6, *The key user journey*)

Qualified people are not paying customers. Between them stands the **journey**: the steps in which the product must
show its value. We simulate it as a chain of conditional probabilities, one simulated person at a time (an
agent-based model, the computational-sociology tool for "what happens when many different people meet this"):

`reached → sets a first habit → keeps it a week → slips (or not) → recovers after the slip → value moment (three weeks
kept) → pays at the day-14 paywall`

* Steps 2–3 and the paywall come from **Ritmo's events per age band** (partial);
* each person's discipline is drawn around their band's completion rate (heterogeneity);
* recovery after a slip is where the **coach** acts: a base rate (assumption) plus a lift scaled by the group's interest
  in coach features from block 6b (research, shape only).
The output that matters is the **difference with and without the coach** per group: the value the coach shows to
that group. Its level is an assumption; its ranking across groups is the evidence.

In [ ]:
evb = u.assign(created=u.create_habit > 0, completed=u.complete_habit > 0, priced=u.view_pricing > 0,
               trial=u.start_trial > 0, paid=u.subscribe > 0)
J = pd.DataFrame({
    "p_create": evb.groupby("age_band").created.mean(),
    "p_week1_given_create": evb[evb.created].groupby("age_band").completed.mean(),
    "completion": (u.complete_habit / u.create_habit.where(u.create_habit > 0)).clip(0, 1).groupby(u.age_band).mean(),
    "p_pay_given_trial": evb[evb.trial].groupby("age_band").paid.mean(),
})
BASE_RECOVER = fact("recover_without_coach", 0.25, 0.45, "share who restart after a slip, no coach", "assumption", "to test")
COACH_LIFT = fact("coach_recovery_lift", 0.05, 0.25, "added restart share from the coach, at average interest", "assumption", "to test: fake door + pilot")
PAY_IF_NO_VALUE = fact("pay_if_no_value_moment", 0.2, 0.4, "paid rate without the value moment, relative", "assumption", "to test")

def journey(band: str, group: str, coach: bool, n: int = 20_000, seed: int = 0) -> dict:
    r, g = J.loc[band], np.random.default_rng(seed)
    created = g.random(n) < r.p_create
    week1 = created & (g.random(n) < r.p_week1_given_create)
    discipline = g.beta(1 + 8 * r.completion, 1 + 8 * (1 - r.completion), n)      # each person: their own chance to keep it
    slips = week1 & (g.random(n) > discipline)
    w = coach_weight.get(group, 1.0) if coach else 0.0
    recover_p = g.uniform(*BASE_RECOVER, n) + w * g.uniform(*COACH_LIFT, n)
    recovered = slips & (g.random(n) < recover_p)
    value = week1 & (~slips | recovered)
    pay_p = r.p_pay_given_trial * np.where(value, 1.0, g.uniform(*PAY_IF_NO_VALUE, n))
    paid = created & (g.random(n) < pay_p)
    return {"set a habit": created.mean(), "kept week 1": week1.mean(), "slipped": slips.mean(),
            "recovered": recovered.mean(), "value moment": value.mean(), "paid": paid.mean()}

GROUP_BAND = {"students": "18-24", "remote_workers": "25-34", "parents": "35-44", "health_55": "55+"}
sims = {}
for grp, band in GROUP_BAND.items():
    for coach in (False, True):
        sims[(grp, "with coach" if coach else "without")] = journey(band, grp, coach)
JS = pd.DataFrame(sims).T
display(JS.round(3))
value_shown = (JS.xs("with coach", level=1) - JS.xs("without", level=1))[["value moment", "paid"]]
for grp, row in value_shown.iterrows():
    fact(f"coach_value_{grp}", *sorted((row["value moment"] * 0.5, row["value moment"] * 1.5)), "added share reaching the value moment",
         "simulated", "journey simulation, block 8c", "shape only")
value_shown.rename(columns=lambda c: "coach adds: " + c).round(3)

In [ ]:
# Route C: matched audience (8b, the bridge) × reach × paid through the simulated journey with the coach × price × 12.
# The simulation only supplies the journey's *shape*; its levels rest on Ritmo's rates and the stated assumptions.
route_c = {}
for grp, band_list in NICHES.age_bands.items():
    aud = sum(M.loc[bd, "aud_bridge"] for bd in band_list) / (len(band_list) if grp == "parents" else 1)
    paid_share = JS.loc[(grp, "with coach"), "paid"]
    rev = aud * np.mean(L.loc["reach_share", ["low", "high"]]) * paid_share * 4.99 * 12
    route_c[grp] = {"qualified (bridge)": aud, "paid via journey": paid_share, "SOM route C (P50-ish)": rev,
                    "SOM route A P50": res.loc[grp, "SOM_p50"]}
pd.DataFrame(route_c).T.round({"qualified (bridge)": -3, "paid via journey": 3, "SOM route C (P50-ish)": -3, "SOM route A P50": -3})

**Your notes · Block 8c** — what you think about this task: what was clear, what was hard or confusing, what you would do differently, and your feedback for the teachers. Your own words, a few lines are enough; your teachers read them in your folder.

*Write below this line (double-click this cell, then Shift+Enter):*

…


## Block 8d · Evidence synthesis: many weak signals, one hypothesis (deck: Part 7)

What we have been doing has names. Four research traditions combine imperfect signals about a shared hypothesis:

| Tradition | Core idea | Here |
|---|---|---|
| **Weak supervision** (Ratner et al., NeurIPS 2016; Snorkel, 2017) | many noisy labeling functions; model their accuracies and dependencies instead of counting votes | reviews and comments → signals of need, spending, switching |
| **Bayesian evidence synthesis** (Jackson et al., 2015) | direct and indirect data joined through one probabilistic model | Ritmo's rate (indirect) + a fake door (direct) → the coach's conversion |
| **Triangulation** (Lawlor, Tilling & Davey Smith, 2016) | agreement counts when the approaches' **biases differ** | conversations, spending, behaviour, simulation |
| **Dempster–Shafer** (Sentz & Ferson, 2002; Shenoy, 2023) | belief and plausibility; ignorance kept apart from evidence against | "this niche fails" ≠ "we do not know yet" |

The synthesis, for market parameters $\theta$ and evidence $D_1, \dots, D_k$:

$$p(\theta \mid D_1, \dots, D_k) \;\propto\; p(\theta)\, p(D_1, \dots, D_k \mid \theta) \;\overset{\text{only if } D_i \perp D_j \mid \theta}{=}\; p(\theta) \prod_i p(D_i \mid \theta)$$

Multiplying related evidence as if it were independent manufactures confidence: two reports quoting the same survey
are one piece of evidence.

In [ ]:
# 1. Weak supervision: labeling functions vote on "this text shows an unmet, paid-for job" (1), "only praise" (0),
#    or abstain (-1). Snorkel's summary: coverage, overlap, conflict; then a vote weighted by estimated accuracy.
LFS = {
    "need_recurring":   (r"\b(?:every (?:day|morning|night)|keep forgetting|always forget|struggl\w*|can.t stick|fall off|relaps\w*)\b", 1),
    "pays_workaround":  (r"\b(?:paid for|pay for|subscribed|bought|premium)\b.{0,80}\b(?:still|but|yet|wish|missing)\b", 1),
    "switching":        (r"\b(?:switch\w*|alternative|looking for|replac\w*|moved from|instead of)\b", 1),
    "asks_feature":     (r"\b(?:please add|wish (?:it|there)|would love|feature request|if only)\b", 1),
    "praise_only":      (r"^(?:(?!\b(?:but|however|wish|problem|bug|crash|forget)\b).)*\b(?:love (?:it|this)|best app|perfect|amazing)\b(?:(?!\b(?:but|however|wish)\b).)*$", 0),
}
texts = pd.concat([rtext, hn.text.fillna("").str.lower()], ignore_index=True)
source = np.r_[np.repeat("review", len(rtext)), np.repeat("hn", len(hn))]
Lam = pd.DataFrame({k: np.where(texts.str.contains(rx, regex=True), v, -1) for k, (rx, v) in LFS.items()})
voted = (Lam >= 0)
summary_lf = pd.DataFrame({"coverage": voted.mean(),
                           "overlap": [(voted[c] & (voted.drop(columns=c).sum(axis=1) > 0)).mean() for c in Lam],
                           "conflict": [(voted[c] & Lam.drop(columns=c).apply(lambda r: ((r >= 0) & (r != LFS[c][1])).any(), axis=1)).mean() for c in Lam]})
majority = Lam.replace(-1, np.nan).mean(axis=1)
mv = np.where(majority.isna(), -1, (majority >= 0.5).astype(int))
acc = pd.Series({c: ((Lam[c] == mv) & (Lam[c] >= 0) & (mv >= 0)).sum() / max(1, ((Lam[c] >= 0) & (mv >= 0)).sum()) for c in Lam})
summary_lf["accuracy vs majority"] = acc                           # one step of what a label model learns properly
logit = sum(np.where(Lam[c] < 0, 0, np.where(Lam[c] == 1, 1, -1)) * np.log(acc[c] / max(1e-3, 1 - acc[c] + 1e-3)) for c in Lam)
label = np.where((Lam >= 0).any(axis=1), (logit > 0).astype(int), -1)
print(f"{len(texts)} texts; labelled by at least one function: {(label >= 0).mean():.2f}; "
      f"positive among labelled: {(label[label >= 0] == 1).mean():.2f}")
summary_lf.round(3)

**Read it:** coverage is how often a function speaks, overlap how often another speaks too, conflict how often they
disagree. Majority voting treats every function as equally good; a label model (Snorkel) learns each one's accuracy
and their correlations. **The labels are about texts, not about the market**: a share of positive texts is not a
share of buyers (who writes is not who buys: block 3b).

In [ ]:
# 2. Bayesian evidence synthesis: the coach's conversion theta, from indirect evidence (Ritmo premium's paid rate r,
#    a different product) and direct evidence (a fake door: k of n visitors pre-ordered; PLACEHOLDER counts), joined
#    through one model: theta = r * delta, where delta (how a coach's rate relates to premium's) is unknown.
rng3 = np.random.default_rng(3)
M_ = 200_000
r_draw = stats.beta(1 + saw.paid.sum(), 1 + len(saw) - saw.paid.sum()).rvs(M_, random_state=rng3) * p_price   # P(pay) per user
delta = rng3.uniform(0.02, 0.6, M_)                                        # prior: wide, an assumption
theta = r_draw * delta
k_fd, n_fd = 9, 400                                                        # the fake door (replace with yours)
wts = stats.binom.pmf(k_fd, n_fd, theta)                                   # likelihood of the direct evidence
wts /= wts.sum()
def q(x, w=None, ps=(5, 50, 95)):
    if w is None:
        return np.percentile(x, ps)
    o = np.argsort(x); c = np.cumsum(w[o]); return np.interp(np.array(ps) / 100, c, x[o])
synth = pd.DataFrame({"prior (Ritmo × delta)": q(theta), "fake door alone (Beta)": stats.beta(1 + k_fd, 1 + n_fd - k_fd).ppf([.05, .5, .95]),
                      "synthesis (both, one model)": q(theta, wts)}, index=["p5", "p50", "p95"]).T
synth["delta p50"] = [np.median(delta), np.nan, q(delta, wts)[1]]
fact("coach_conversion_synth", *q(theta, wts, (5, 95)), "share of visitors who pre-order the coach", "partial",
     "evidence synthesis: Ritmo (indirect) + fake door (direct, PLACEHOLDER 9/400)")
synth.round(4)

In [ ]:
# 3. Triangulation: do sources with DIFFERENT biases agree that remote workers are a better first niche than students?
tri = pd.DataFrame([
    ("conversations (HN)",     "who posts: tech-heavy, vocal, few over 55",      voice.loc["remote_workers", "share_of_voice"] > voice.loc["students", "share_of_voice"]),
    ("app reviews",            "who reviews: extremes; iPhone users in the US",  coach_weight.get("remote_workers", 1) >= coach_weight.get("students", 1)),
    ("Ritmo behaviour",        "already users of a habit app (selection)",       saw[saw.age_band.isin(["25-34", "35-44"])].paid.mean() > saw[saw.age_band == "18-24"].paid.mean()),
    ("Eurostat spending",      "definition: any health or fitness app",          b["25-34"] > b["18-24"]),
    ("journey simulation",     "assumptions; an agent-based model",              JS.loc[("remote_workers", "with coach"), "paid"] > JS.loc[("students", "with coach"), "paid"]),
], columns=["source", "its main bias", "remote workers ahead?"])
tri["remote workers ahead?"] = tri["remote workers ahead?"].astype(bool)
print(f"{tri['remote workers ahead?'].sum()} of {len(tri)} sources agree; their biases differ, so agreement says more than five copies of one survey would.")
tri

In [ ]:
# 4. Dempster–Shafer on the frame {viable, not}: each source puts mass on "viable", "not", or the whole frame
#    ("we do not know"). The masses are judgements (assumptions), written down so they can be argued with.
def ds_combine(m1: dict, m2: dict) -> dict:
    """Dempster's rule on the frame {V, N}; 'VN' is ignorance. Returns the combined masses and the conflict K."""
    sets = {"V": {"V"}, "N": {"N"}, "VN": {"V", "N"}}
    out, K = {"V": 0.0, "N": 0.0, "VN": 0.0}, 0.0
    for a_, ma in ((k, m1[k]) for k in sets):                       # masses only: "K" is a report, not a mass
        for b_, mb in ((k, m2[k]) for k in sets):
            inter = sets[a_] & sets[b_]
            if not inter:
                K += ma * mb
            else:
                out[next(k for k, v_ in sets.items() if v_ == inter)] += ma * mb
    return {k: v_ / (1 - K) for k, v_ in out.items()} | {"K": K}
EVID = {"conversations": {"V": 0.30, "N": 0.00, "VN": 0.70}, "Eurostat budget": {"V": 0.20, "N": 0.05, "VN": 0.75},
        "Ritmo bridge": {"V": 0.40, "N": 0.10, "VN": 0.50}, "fake door (placeholder)": {"V": 0.15, "N": 0.45, "VN": 0.40}}
m, rows = {"V": 0.0, "N": 0.0, "VN": 1.0}, []
for name, e in EVID.items():
    m = ds_combine(m, e)
    rows.append({"after": name, "Bel(viable)": m["V"], "Pl(viable)": m["V"] + m["VN"], "ignorance": m["VN"], "conflict K": m["K"]})
twice = ds_combine(ds_combine(EVID["conversations"], EVID["conversations"]), EVID["Eurostat budget"])
print(f"double counting: conversations combined twice gives Bel(viable) = {twice['V']:.2f}, against "
      f"{ds_combine(EVID['conversations'], EVID['Eurostat budget'])['V']:.2f} once: the same evidence, more confidence")
pd.DataFrame(rows).set_index("after").round(3)

**Read it:** $\text{Bel}$ is what the evidence supports, $\text{Pl} = 1 - \text{Bel}(\text{not})$ what it does not
rule out; the gap is **ignorance**, which only new evidence closes. Dempster's rule,
$m_{12}(A) = \frac{1}{1-K}\sum_{B \cap C = A} m_1(B)\,m_2(C)$ with conflict $K = \sum_{B \cap C = \varnothing} m_1(B)\,m_2(C)$,
assumes the sources are **distinct**: combine the same evidence twice and belief grows without new information
(Shenoy, 2023). The method for your project, named: **Bayesian evidence synthesis for niche discovery**, with weak
supervision to extract the signals and triangulation to audit their agreement.

**Your notes · Block 8d** — what you think about this task: what was clear, what was hard or confusing, what you would do differently, and your feedback for the teachers. Your own words, a few lines are enough; your teachers read them in your folder.

*Write below this line (double-click this cell, then Shift+Enter):*

…


## Block 9 · Niche variants (deck: Part 8)

A niche is more than an age band: **who × which job × at what price × through which channel**. Generate the variants,
score every one with the same fused model, and keep the ones that are good **in the pessimistic case** (P10), not just
on average.

In [ ]:
variants = []
for niche in NICHES.index:
    for price, arm in ((3.99, "pricing_a"), (4.99, "pricing_b"), (7.99, "pricing_b")):
        extra = {}
        if price == 7.99:   # beyond the test: the simulation's shape (relative to 4.99) adjusts the observed rate
            bands = NICHES.loc[niche, "age_bands"]
            rel = np.mean([sim.loc[b, "7.99"] / sim.loc[b, "4.99"] for b in bands])
            extra = {"sim_adjust_799": lambda n, rel=rel: rng.uniform(rel * 0.7, min(1, rel * 1.3), n)}
        a = route_a(niche, price, arm, extra=extra)
        p10, p50, p90 = np.percentile(a.revenue_eur, [10, 50, 90])
        variants.append(dict(niche=niche, price=price, uses_simulation=bool(extra), p10=p10, p50=p50, p90=p90))
V = pd.DataFrame(variants).sort_values("p10", ascending=False)
V.round({"p10": -2, "p50": -2, "p90": -2})

**Your notes · Block 9** — what you think about this task: what was clear, what was hard or confusing, what you would do differently, and your feedback for the teachers. Your own words, a few lines are enough; your teachers read them in your folder.

*Write below this line (double-click this cell, then Shift+Enter):*

…


## Block 10 · Steer: research the input that matters, then update (deck: Part 8)

Two ways to buy information:

1. **Desk research** — a deep-research call (Perplexity, ChatGPT, Gemini or Claude) with `deep_research_prompt.md`,
   asking for the one input at the top of block 8's list. You **check its sources** before it goes in, as `research`.
2. **A fake-door test** — a landing page for the top two variants; the sign-ups update the paid rate by Bayes.

Below, both are wired in. The numbers in `new_research` and `fake_door` are placeholders: replace them with what you find.

In [ ]:
DEEP_RESEARCH_PROMPT = (EX / "deep_research_prompt.md").read_text() if (EX / "deep_research_prompt.md").exists() else ""
print(DEEP_RESEARCH_PROMPT[:600] or "deep_research_prompt.md not found")

# 1. desk research: replace the assumption with a checked range (PLACEHOLDER numbers — put yours here, with the URL)
new_research = {"habit_app_users_share": (0.05, 0.08, "PLACEHOLDER: survey or app-intelligence source, URL, date")}
for k, (lo, hi, src_) in new_research.items():
    fact(k, lo, hi, "share of adults using a habit/self-improvement app", "research", src_)

# 2. fake door for the top two variants (PLACEHOLDER counts — a real test replaces them)
top2 = V.head(2)
fake_door = {top2.iloc[0].niche: (400, 9), top2.iloc[1].niche: (400, 5)}    # visits, paid pre-orders
L = ledger()
before = V.set_index(["niche", "price"]).loc[[tuple(r) for r in top2[["niche", "price"]].values]]
after = []
for _, r in top2.iterrows():
    visits, orders = fake_door[r.niche]
    d = stats.beta(1 + orders, 1 + visits - orders)
    a = route_a(r.niche, r.price, "pricing_b", extra={})
    a["paid_rate"] = d.rvs(len(a), random_state=rng); a["see_price"] = 1.0       # the fake door measures both at once
    a["revenue_eur"] = a.drop(columns=["sam_eur", "revenue_eur"]).prod(axis=1) * r.price * 12
    after.append(dict(niche=r.niche, price=r.price, p10=np.percentile(a.revenue_eur, 10), p50=np.median(a.revenue_eur), p90=np.percentile(a.revenue_eur, 90)))
pd.concat({"before": before.reset_index()[["niche", "price", "p10", "p50", "p90"]], "after": pd.DataFrame(after)}).round({"p10": -2, "p50": -2, "p90": -2})

In [ ]:
# The ledger is the proof: every number, its kind and its source. Export it with the estimate.
L = ledger()
display(L.kind.value_counts())
L.to_csv("niche_ledger.csv"); V.to_csv("niche_variants.csv")
L[["low", "high", "unit", "kind", "source"]]

**Your notes · Block 10** — what you think about this task: what was clear, what was hard or confusing, what you would do differently, and your feedback for the teachers. Your own words, a few lines are enough; your teachers read them in your folder.

*Write below this line (double-click this cell, then Shift+Enter):*

…


## Block 11 · Lab: your own product (deck: Lab)

Follow `examples/session-09/CHEATSHEET.pdf`; in short:

1. Copy `examples/session-09/niche_brief.md` into your project; fill in the product, three to five candidate niches and
   their jobs.
2. Give your agent (Claude Code, Codex, or a chat) `niche_brief.md` + `analyst_agent.md` and ask for the ledger: one
   `fact()` per number it needs, each with a source or marked `assumption`.
3. Run blocks 1, 3b, 6 and 8 with your niches (change `NICHES`, `CUES` and the population bands); reweight your own
   sample, if you have one, and write down whom it cannot speak for.
4. In your journal: the top variant by P10, the input that moves it most, and what you will do to research it.

**Your notes · Block 11** — what you think about this task: what was clear, what was hard or confusing, what you would do differently, and your feedback for the teachers. Your own words, a few lines are enough; your teachers read them in your folder.

*Write below this line (double-click this cell, then Shift+Enter):*

…


## If there is time

Extra work for when the session runs ahead of plan; each takes about 10 minutes and stands alone. Your teacher picks one, or try them at home.

* **Price it: 4.99 against 7.99 per age band**: Compare 4.99 and 7.99 per age band by posterior draws of revenue per viewer (notebook block 2b), and mark the [Today's notebook · Block 2b (Colab)](https://colab.research.google.com/drive/1uzleW63_UJhIzcSXPScCyRMA4r7WeJV8)
* **Rake our group on two margins**: Rake our group on the age and education margins (notebook block 3b), compare with post-stratification on the [Today's notebook · Block 3b (Colab)](https://colab.research.google.com/drive/1uzleW63_UJhIzcSXPScCyRMA4r7WeJV8)
* **A sixth source for the triangle**: Add one source with its own bias to the triangulation in notebook block 8d and say whether it strengthens the [Today's notebook · Block 8d (Colab)](https://colab.research.google.com/drive/1uzleW63_UJhIzcSXPScCyRMA4r7WeJV8)
* **Check the agent three times**: Ask a chat the same analysis question three times with `analyst_agent.md`, run `check()` on each answer [examples/session-09 · instruction files and data](https://drive.google.com/drive/folders/1TyjgCL-ntWJ2zhoHj6zVukyMB6R_NSfB)
